In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8cd319b4f2187b6b29bb69603a96460fc325a975"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8cd319b4f2187b6b29bb69603a96460fc325a975/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Synthetic Regression Data


Before we can train a model we need data.
Real datasets are what we ultimately care about,
but they conflate three separate sources of failure:
a misspecified model, a flawed optimization algorithm,
and pathological data.
When a method performs poorly on real data,
all three explanations remain on the table at once.
*Synthetic data* removes this ambiguity by construction.
If we know the data-generating process exactly
(the true weights $\mathbf{w}^*$, the true bias $b^*$,
and the noise distribution),
then any *systematic* failure to recover them
(beyond the irreducible noise) points to the algorithm
or implementation.
This is why synthetic datasets are the first test
for any new learning method.
We confirm that it solves a problem with a known answer
before we ever hand it a real one.

In [1]:
%matplotlib inline
from d2l import jax as d2l
import jax
from jax import numpy as jnp
import numpy as np
import random
import tensorflow as tf

## Generating the Dataset

For this example, we will work in low dimension
for succinctness.
The following code snippet generates 1000 examples
with 2-dimensional features drawn 
from a standard normal distribution.
The resulting design matrix $\mathbf{X}$
belongs to $\mathbb{R}^{1000 \times 2}$. 
We generate each label by applying 
a *ground truth* linear function, 
corrupting them via additive noise $\boldsymbol{\epsilon}$, 
drawn independently and identically for each example:

$$\mathbf{y}= \mathbf{X} \mathbf{w}^* + b^* + \boldsymbol{\epsilon}.$$

For convenience we assume that $\boldsymbol{\epsilon}$ is drawn 
from a normal distribution with mean $\mu= 0$ 
and standard deviation $\sigma = 0.01$.
We put the generation code in the `__init__` method of a subclass
of `d2l.DataModule` (introduced in that section),
calling `save_hyperparameters()` so that every constructor argument
(the parameters `w` and `b`, the noise level, the split sizes, and
`batch_size`) is stored as an attribute and the dataset stays
introspectable.

In [2]:
class SyntheticRegressionData(d2l.DataModule):
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32, key=None):
        super().__init__()
        self.save_hyperparameters()
        # Resolve the key at call time rather than reusing a key in the signature.
        key = jax.random.key(0) if key is None else key
        n = num_train + num_val
        key1, key2 = jax.random.split(key)
        self.X = jax.random.normal(key1, (n, w.shape[0]))
        eps = jax.random.normal(key2, (n, 1)) * noise
        self.y = d2l.matmul(self.X, d2l.reshape(w, (-1, 1))) + b + eps

Below, we set the true parameters to $\mathbf{w}^* = [2, -3.4]^\top$ and $b^* = 4.2$.
Later, we can check our estimated parameters against these *ground truth* values.

In [3]:
data = SyntheticRegressionData(w=d2l.tensor([2, -3.4]), b=4.2)

Each row of `data.X` is a feature vector in $\mathbb{R}^2$ and each row of `data.y` is a scalar label. Let's have a look at the first entry.

In [4]:
print('features:', data.X[0],'\nlabel:', data.y[0])

features: [ 1.0040143 -0.9063372] 
label: [9.265151]


## Reading the Dataset

Training machine learning models often requires multiple passes over a dataset, 
grabbing one minibatch of examples at a time. 
This data is then used to update the model. 
To illustrate how this works, we 
implement the `get_dataloader` method, 
registering it in the `SyntheticRegressionData` class via `add_to_class` (introduced in that section).
It takes a batch size, a matrix of features,
and a vector of labels, and generates minibatches of size `batch_size`.
As such, each minibatch consists of a tuple of features and labels. 
Note that we need to be mindful of whether we're in training or validation mode: 
in the former, we will want to read the data in random order, 
whereas for the latter, being able to read data in a pre-defined order 
may be important for debugging purposes.

In [5]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))
        # The examples are read in random order
        random.shuffle(indices)
    else:
        indices = list(range(self.num_train, self.num_train+self.num_val))
    for i in range(0, len(indices), self.batch_size):
        batch_indices = d2l.tensor(indices[i: i+self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]

To build some intuition, let's inspect the first minibatch of
data. Each minibatch of features provides us with both its size and the dimensionality of input features.
Likewise, our minibatch of labels will have a matching shape given by `batch_size`.

In [6]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: (32, 2) 
y shape: (32, 1)


Iterating over `data.train_dataloader()` yields distinct minibatches
until the dataset is exhausted (try it).
Writing the loader by hand makes every step explicit,
but it costs us in three ways:
all of the data must fit in memory, the iteration is single-threaded
Python looping over indices, and there is no prefetching to overlap
data loading with computation on the previous batch.
The data loaders built into a deep learning framework fix all three.
They run several worker processes in parallel, prefetch the next batch
while the current one trains, and stream from sources such as files,
network streams, or generators that produce data on the fly.
We now switch to the framework's built-in loader,
which presents an identical interface to the caller.

## Concise Implementation of the Data Loader

Rather than writing our own iterator,
we can call the existing API in a framework to load data.
As before, we need a dataset with features `X` and labels `y`. 
Beyond that, we set `batch_size` in the built-in data loader 
and let it take care of shuffling examples  efficiently.

JAX is all about NumPy like API with device acceleration and the functional
transformations, so at least the current version doesn’t include data loading
methods. With other  libraries we already have great data loaders out there,
and JAX suggests using them instead. Here we will grab TensorFlow’s data loader,
and modify it slightly to make it work with JAX.

In [7]:
class TensorFlowDataLoader:
    """Expose a tf.data.Dataset as re-iterable NumPy batches."""
    def __init__(self, dataset):
        self.dataset = dataset

    def __iter__(self):
        return self.dataset.as_numpy_iterator()

    def __len__(self):
        return len(self.dataset)

@d2l.add_to_class(d2l.DataModule)
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    # Use TensorFlow's data loader. JAX and Flax do not provide data-loading
    # functionality. `drop_remainder=train` keeps every
    # *training* minibatch the same shape, so a `@jax.jit`'d step
    # function compiles once per epoch instead of recompiling for the
    # smaller last batch.
    shuffle_buffer = tensors[0].shape[0] if train else 1
    dataset = tf.data.Dataset.from_tensor_slices(tensors).shuffle(
        buffer_size=shuffle_buffer).batch(
            self.batch_size, drop_remainder=train)
    return TensorFlowDataLoader(dataset)

In [8]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

The new data loader behaves just like the previous one, except that it is more efficient and has some added functionality.

In [9]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: (32, 2) 
y shape: (32, 1)


For instance, the data loader provided by the framework API 
supports the built-in `__len__` method, 
so we can query its length, 
i.e., the number of batches.

In [10]:
len(data.train_dataloader())

31

With 1000 training examples and a batch size of 32, we expect
$\lceil 1000 / 32 \rceil = 32$ batches: 31 full ones and a final
partial batch of 8 examples.
Note also that the built-in training loader *reshuffles* the examples at
the start of every epoch, just as our hand-rolled loader drew a fresh
random order on each call; exercise 8 of that section
asks why this reshuffling matters.

You may notice that the JAX loader reports 31 batches rather than 32.
This is because `get_tensorloader` passes `drop_remainder=True` when
training: the final partial batch of 8 examples is discarded.
We do this so that every training minibatch has an identical shape,
which keeps a `@jax.jit`-compiled training step from being recompiled
for the differently sized last batch (a recompilation that can cost
minutes per epoch on larger datasets). The price is that we drop a
handful of examples each epoch, which is negligible here. A loader that
keeps the partial batch would report 32.

## Summary

Synthetic data lets us check the recovered parameters against the truth
we fixed: because we chose $\mathbf{w}^*$ and $b^*$ ourselves, we can see
after training whether the estimates agree, which makes such datasets
the first place to validate any new algorithm.
The `SyntheticRegressionData` class introduced here packages this
data-generating process as a `DataModule` subclass, separating *where
the batches come from* from *how a model consumes them*.
Along the way we implemented the same `get_dataloader` protocol twice:
a transparent hand-rolled iterator that is easy to read but loads
everything in memory and loops in Python, and a framework-native loader
that shuffles, prefetches, and parallelizes for us.
The hand-rolled version is there to teach; the framework version is what
we use from here on.


## Exercises

1. When the number of examples is not divisible by the batch size, the loaders above keep the final partial batch. What does PyTorch's `drop_last` argument (and its TensorFlow counterpart, `batch(..., drop_remainder=...)`) do, and when would you want to enable it?
1. Suppose that we want to generate a huge dataset, where both the size of the parameter vector `w` and the number of examples `num_examples` are large.
    1. What happens if we cannot hold all data in memory?
    1. How would you shuffle the data if it is held on disk? Your task is to design an *efficient* algorithm that does not require too many random reads or writes. Hint: [pseudorandom permutation generators](https://en.wikipedia.org/wiki/Pseudorandom_permutation) allow you to design a reshuffle without the need to store the permutation table explicitly [@Naor.Reingold.1999]. 
1. Implement a data generator that produces new data on the fly, every time the iterator is called. 
1. **(Reproducibility.)** How would you design a random data generator that
   produces the *same* dataset every time it is called? Libraries differ here:
   some expose a single global seed, while others thread an explicit random
   state through every draw. For the threaded style, explain why fixing that
   state up front makes the result reproducible, and why re-using the *same*
   state for both $\mathbf{X}$ and $\boldsymbol{\epsilon}$ (rather than
   advancing it between the two draws) would be a bug.
1. **(Signal-to-noise and recovery.)** Vary the noise standard deviation `noise`
   over $\{0.001, 0.01, 0.1, 0.5, 1.0\}$. After fitting a linear model on each
   dataset (using the code from that section or
   that section), how closely does the estimate
   $\hat{\mathbf{w}}$ match the true $\mathbf{w}^* = [2, -3.4]^\top$? Plot the
   error $\|\hat{\mathbf{w}} - \mathbf{w}^*\|_2$ as a function of $\sigma$. How do
   you expect it to scale with $\sigma$ and with the number of training examples,
   and does the experiment agree?

[Discussions](https://d2l.discourse.group/t/17975)